In [ ]:
#@title (1) p.142-146 本のままではColabで動かない確認

import cv2

# カメラの初期化
cap = cv2.VideoCapture(0)

# 静止画の取得
ret, frame = cap.read()

# 画像を保存
if ret:
  cv2.imwrite('img.jpg', frame)

# 解放処理
cap.release()

# 独自クラス ColabCap を作った. これを cap 名でインスタンス化し本 p.146以降と同等のコードにする

In [ ]:
#@title (2) 独自クラス ColabCap (本の cap = cv2.VideoCapture(0) の代用)

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2

class ColabCap:

  _js = '''
    let video = document.createElement('video');
    let canvas = document.createElement('canvas');
    let stream = null;

    async function createDom() {
      if (stream) return;
      stream = await navigator.mediaDevices.getUserMedia({ video: true });
      video.srcObject = stream;
      await video.play();
    }

    async function removeDom() {
      await stream.getVideoTracks()[0].stop();
      video = null;
      stream = null;
      canvas = null;
    }

    async function cap(quality, waitSec) {
      if (!stream) await createDom();
      await new Promise((resolve) => setTimeout(resolve, waitSec * 10**3));
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      return canvas.toDataURL('image/jpeg', quality);
    }
  '''

  def __init__(self, quality=0.8, first_wait_sec=0.25):
    self.quality = quality
    self.first_wait_sec = first_wait_sec
    display(Javascript(ColabCap._js))

  def read(self):
    try:
      data = eval_js(f'cap({ self.quality }, { self.first_wait_sec })')
      self.first_wait_sec = 0
      image_bytes = b64decode(data.split(',')[1])
      jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
      return True, cv2.imdecode(jpg_as_np, flags=1)
    except Exception as err:
      print(str(err))
      return False, None

  def release(self):
    eval_js('removeDom()')


In [ ]:
#@title (3) 独自クラス ColabCap のテスト。本 p.146 (5-1-2) と同様、静止画を撮影・保存

# 実行前に1回ColabCapを定義したセルを実行する (Colab特有)
# 初回実行時はたいていカメラ利用許可がまだなく、エラーになる
# 2回目の実行でカメラ利用許可の確認ダイアログが出たらOKする
# 動作確認済 2026.4.29 Chrome 146.0, Firefox 149.0

# カメラの初期化
# cap = cv2.VideoCapture(0) # 本
cap = ColabCap() # Colab版

# 静止画の取得
ret, frame = cap.read()

# 画像を保存
if ret:
  cv2.imwrite('img.jpg', frame)

# 解放処理
cap.release()


In [ ]:
#@title (4) 本 p.148 (5-1-3) と同様、静止画を撮影・保存・表示

# 実行前に1回 ColabCap を定義したセルを実行する (Colab特有)
# 初回実行時はたいていカメラ利用許可がまだなく、エラーになる
# 2回目の実行でカメラ利用許可の確認ダイアログが出たらOKする
# 動作確認済 2026.4.29 Chrome 146.0, Firefox 148.0

# 画像表示はColab用ライブラリを使う
from google.colab.patches import cv2_imshow

# カメラの初期化
cap = ColabCap()

# 静止画の取得
ret, frame = cap.read()

# 画像を保存
if ret:
  cv2.imwrite('img.jpg', frame)

# 解放処理
cap.release()

# 結果表示
cv2_imshow(frame)


In [ ]:
#@title (5) 本 p.149〜153 (5-2) に近い結果を得る (動画撮影・保存)
# この方法では滑らかな動画にならない. 1枚1枚をブラウザで取得しColabへ送っているため
# 動画表示も遅いので省略

# 実行前に1回 ColabCap を定義したセルを実行する (Colab特有)
# 初回実行時はたいていカメラ利用許可がまだなく、エラーになる
# 2回目の実行でカメラ利用許可の確認ダイアログが出たらOKする
# 動作確認済 2026.4.29 Chrome 146.0, Firefox 148.0

import cv2
import time

# カメラの初期化
cap = ColabCap()

# 撮影条件
frame_rate = 10 # Colabでは10fps位が限界. sleepなしで限界を確かめ設定するのが現実的
duration = 10
interval = 1 / frame_rate
frame_count = int(duration / interval)

# 動画保存条件
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('movie.mp4', fourcc, frame_rate, (640, 480))

# 動画撮影
for i in range(frame_count):
  ret, frame = cap.read()
  if not ret:
    break
  out.write(frame)

  # 再生すると極端に遅くなる (出力欄をつど消去するため)
  # cv2_imshow(frame)
  # output.clear()

  # time.sleep(interval)
  # Colabではフレームレートを高くできないので不要

cap.release()
out.release()


In [ ]:
#@title (6) 本 p.154〜157 (5-3) に近い結果を得る (タイムラプス動画撮影・保存)

# 実行前に1回 ColabCap を定義したセルを実行する (Colab特有)
# 初回実行時はたいていカメラ利用許可がまだなく、エラーになる
# 2回目の実行でカメラ利用許可の確認ダイアログが出たらOKする
# 動作確認済 2026.3.27 Chrome 146.0, Firefox 148.0

import cv2
import time
from google.colab.patches import cv2_imshow
from google.colab import output

# カメラの初期化
cap = ColabCap()

# 撮影条件
frame_rate = 10
duration = 60
interval = 2
frame_count = int(duration / interval)

# 動画保存条件
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('movie_timelapse.mp4', fourcc, frame_rate, (640, 480))

# 動画撮影
for i in range(frame_count):
  ret, frame = cap.read()
  if not ret:
    break
  out.write(frame)

  cv2_imshow(frame)

  time.sleep(interval) # 実際はintervalより少し長めに待機してしまう. ブラウザの処理と通信が入るため
  output.clear()

cap.release()
out.release()


In [ ]:
#@title (7) 本 p.159 (5-4-1) と同様、Colabにある画像を表示

import cv2
from google.colab.patches import cv2_imshow

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像の表示
cv2_imshow(img)

In [ ]:
#@title (7) 本 p.160〜161 (5-4-2) と同様、画像をセピア色に変換

from google.colab.patches import cv2_imshow
import cv2
import numpy as np

def apply_color_tone(img):
  """画像に色効果を適用する関数"""

  # セピア調にするカラー変換行列
  # 注：0.xxxの先頭ゼロは略せる 工学系のよくある記法
  # 注：インデント浅くした

  sepia_filter = np.array([
    [.272, .534, .131],
    [.349, .686, .168],
    [.393, .769, .189]
  ])
  applied_img = cv2.transform(img, sepia_filter)

  # 値を0〜255の範囲に変更
  applied_img = np.clip(applied_img, 0, 255).astype(np.uint8)

  return applied_img

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_color_tone(img)

# 画像の表示
cv2_imshow(applied_img)

# 画像の保存
cv2.imwrite('img_out.jpg', applied_img)


In [ ]:
#@title (8) 本 p.163 (5-4-3) と同様、好きな色変換を試す

from google.colab.patches import cv2_imshow
import cv2
import numpy as np

def apply_color_tone(img):
  """画像に色効果を適用する関数"""

  # セピア以外の変換の例 本p.166 ⑥全体的に明るくする
  filter = np.array([
    [3, 0, 0],
    [0, 3, 0],
    [0, 0, 3]
  ])
  applied_img = cv2.transform(img, filter)

  # 値を0〜255の範囲に変更
  applied_img = np.clip(applied_img, 0, 255).astype(np.uint8)

  return applied_img

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_color_tone(img)

# 画像の表示
cv2_imshow(applied_img)

# 画像の保存
cv2.imwrite('img_out.jpg', applied_img)


In [ ]:
#@title (9) 本 p.166〜167 (5-4-4) と同様、画像のエッジを強調

from google.colab.patches import cv2_imshow
import cv2

def apply_edges(img):
  """エッジを検出して元画像に重ね描きする関数"""

  # グレースケールに変換
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  # エッジを検出
  edges = cv2.Canny(gray, 100, 200)

  # エッジを黒色で描画
  img[edges == 255] = (0, 0, 0)

  return img

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_edges(img)

# 画像の表示
cv2_imshow(applied_img)


In [ ]:
#@title (10) 本 p.169 (5-4-5) と同様、画像のエッジを太くする

import numpy as np
from google.colab.patches import cv2_imshow
import cv2

def apply_edges(img):
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  edges = cv2.Canny(gray, 100, 200)

  # エッジを太くする
  kernel = np.ones((3, 3), np.uint8)
  edges_dilated = cv2.dilate(edges, kernel, iterations=1)

  # エッジを黒色で描画
  img[edges_dilated == 255] = (0, 0, 0)

  return img

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_edges(img)

# 画像の表示
cv2_imshow(applied_img)


In [ ]:
#@title (11) 本 p.170 (5-4-6) と同様、画像をぼかす

from google.colab.patches import cv2_imshow
import cv2

def apply_blur(img):
  """画像全体にぼかし効果を追加する関数"""
  kernel = (15, 15)
  return cv2.GaussianBlur(img, kernel, 0)

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_blur(img)

# 画像の表示
cv2_imshow(applied_img)


In [ ]:
#@title 第12回予定 (1) <br>本の方法 (画像1コマ1コマを蓄積) で作成した動画は、そのままではColab内で再生できない<br>前回の ↑ セル(5)または(6)を実行して適当にmovie.mp4を作って確認

from IPython.display import display
from base64 import b64encode

mp4 = open('movie.mp4', "rb").read()
b64 = b64encode(mp4).decode()

# GitHubで正しく表示できるようHTMLタグを使わない
display(Javascript(f"""
  const video = document.createElement("video");
  video.autoplay = true;
  video.controls = true;
  const source = document.createElement("source");
  source.src = "data:video/mp4;base64,{b64}";
  video.appendChild(source);
  document.querySelector("#output-area").appendChild(video);
"""));


In [ ]:
#@title 第12回予定 (2) <br>FFmpegで変換すると再生できる

from IPython.display import display
from base64 import b64encode
import subprocess
import os

# FFmpegで変換し、最後に一時ファイルを消す
# GitHubで正しく表示できるよう「!コマンド $変数」は使わない
tmp_file = "tmp.mp4"
subprocess.run(["ffmpeg", "-i", "movie.mp4", tmp_file, "-y", "-loglevel", "0"])
mp4 = open(tmp_file, "rb").read()
os.remove(tmp_file)
b64 = b64encode(mp4).decode()

# GitHubで正しく表示できるようHTMLタグを使わない
display(Javascript(f"""
  const video = document.createElement("video");
  video.autoplay = true;
  video.controls = true;
  const source = document.createElement("source");
  source.src = "data:video/mp4;base64,{b64}";
  video.appendChild(source);
  document.querySelector("#output-area").appendChild(video);
"""));


In [ ]:
#@title 第12回予定 (3) <br>(2)を独自関数 play_video 化

from IPython.display import display
from base64 import b64encode
import subprocess
import os

def play_video(movie_file, video_height=240):
  if not os.path.isfile(movie_file):
    print(f"Error: file '{movie_file}' not found")
    return

  # FFmpegで変換し、最後に一時ファイルを消す
  # GitHubで正しく表示できるよう「!コマンド $変数」は使わない
  tmp_file = "tmp.mp4"
  subprocess.run(["ffmpeg", "-i", movie_file, tmp_file, "-y", "-loglevel", "0"])
  mp4 = open(tmp_file, "rb").read()
  os.remove(tmp_file)

  # FFmpegの変換結果を標準出力で扱えれば一時ファイル不要になるが、それは難しい
  # https://qiita.com/rougemeilland/items/d1c2514caaa7682ff683

  # GitHubで正しく表示できるようHTMLタグを使わない
  display(Javascript(f"""
    const video = document.createElement("video");
    video.height = "{video_height}";
    video.autoplay = true;
    video.controls = true;
    const source = document.createElement("source");
    source.src = "data:video/mp4;base64,{b64encode(mp4).decode()}";
    video.appendChild(source);
    document.querySelector("#output-area").appendChild(video);
  """));

print("再生準備完了")

In [ ]:
#@title 第12回予定 (4)<br>独自関数 play_video のテスト

play_video("movie.mp4", 320)

In [ ]:
#@title 第12回予定 (5)<br>前回、予告的に紹介した動画保存用の独自関数 VideoWriter (その後修正あり)<br>- 本の cv2.VideoWriter の代用、第6章の準備も兼ねる

from google.colab.output import eval_js
from IPython.display import display, Javascript
from base64 import b64decode

def VideoWriter(out_file, duration, frame_rate, wh):
	# Colabで動画撮影・保存
	# 本 p.151 "cv2.VideoWriter" の代用
	# 通常の動画撮影用, タイムラプス撮影はできない

	_js = """
		const outputArea = document.querySelector("#output-area");
		const video = document.createElement("video");
		const info = document.createElement("p");
		let blob = null
		let durationMsec = null

		async function record(durationSec, frameRate, width, height) {
			durationMsec = durationSec * 10**3;
			const browser = navigator.mediaDevices;
			const supported = browser.getSupportedConstraints();
			if (!supported.width || !supported.height || !supported.frameRate) {
				throw new Error("Browser is not supported for width, height or frameRate.");
			}
			const stream = await browser.getUserMedia({
				video: {
					width,
					height,
					frameRate,
						// Chrome: width/heightの挙動が変(v146) ex.720x480を指定すると1024x684になる
						// Firefox: frameRateの挙動が変(v149) ex.15を指定すると60になる
						// c.f. https://developer.mozilla.org/ja/docs/Web/API/MediaDevices/getUserMedia
				},
				audio: false,
			});
			startDisplay(stream);
			const result = await recordCore(stream, {
				// mimeType: "video/mp4" // error on Firefox
			});
			stopDisplay();
			return result;
		}

		async function recordCore(stream, options) {
			const mediaRecorder = new MediaRecorder(stream, options);
			mediaRecorder.addEventListener(
				"dataavailable", e => {
					// fired on mediaRecorder.stop
					blob = e.data;
				}
			);
			mediaRecorder.start();
			await new Promise(resolve => setTimeout(resolve, durationMsec));
			mediaRecorder.stop();
			// wait data
			await new Promise(resolve => {
				const tid = setInterval(() => {
					if (blob == null) return;
					clearInterval(tid);
					resolve();
				}, 10);
			});
			stream.getVideoTracks()[0].stop();
			return await blobToDataURL(blob);
		}

		function startDisplay(stream) {
			message("録画中･･･");
			outputArea.appendChild(info);
			outputArea.appendChild(video)
			video.srcObject = stream;
			video.play();
		}

		function stopDisplay() {
			video.pause();
			video.parentNode.removeChild(video);
			message("録画終了。転送処理中･･･");
		}

		// https://stackoverflow.com/questions/23150333
		async function blobToDataURL(blob) {
			return await new Promise((resolve, reject) => {
				const reader = new FileReader();
				reader.onload = () => resolve(reader.result);
				reader.onerror = () => reject(reader.error);
				reader.onabort = () => reject(new Error("Read aborted"));
				reader.readAsDataURL(blob);
			});
		}

		function message(str) {
			info.textContent = str;
		}

		function clearInfo() {
			info.parentNode.removeChild(info);
		}
	"""

	try:
		display(Javascript(_js))
		data = eval_js(f"""record({duration}, {frame_rate}, {wh[0]}, {wh[1]})""")
		bin = b64decode(data.split("base64,")[1])
		eval_js("""message("動画準備中･･･")""")
		open(out_file, "wb").write(bin)
		eval_js("clearInfo()")
		return True

	except Exception as err:
		print(f"Error: {str(err)}")
		return False

print("録画準備完了")


In [ ]:
#@title 第12回予定 (6) 2つの独自関数 VideoWriter, play_video をまとめてテスト<br>動画の撮影・保存・再生を簡単に行える

out_file = "movie.mp4"
duration = 10
frame_rate = 30

if VideoWriter(out_file, duration, frame_rate, (640, 480)):
  play_video(out_file, 320)


In [ ]:
#@title 第12回予定 (7) 本 p.172〜173 (5-5-1) と同様に動画の色変換・ぼかし ＋ 自動再生

import cv2
import numpy as np

def apply_color_tone(img):
  """画像に色効果を適用する関数"""

  # セピア以外の変換の例 本p.166 ⑥全体的に明るくする
  filter = np.array([
    [3, 0, 0],
    [0, 3, 0],
    [0, 0, 3]
  ])
  applied_img = cv2.transform(img, filter)

  # 値を0〜255の範囲に変更
  applied_img = np.clip(applied_img, 0, 255).astype(np.uint8)

  return applied_img

def apply_blur(img):
  """画像全体にぼかし効果を追加する関数"""
  kernel = (15, 15)
  return cv2.GaussianBlur(img, kernel, 0)

# 動画ファイルの読み込み
cap = cv2.VideoCapture("movie.mp4")

# フレームレートの取得
frame_rate = cap.get(cv2.CAP_PROP_FPS)

# 注：本の55行目 interval はどこからも呼ばれず不要

# 動画の幅と高さを取得
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 動画保存条件
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("movie_1_edited.mp4", fourcc, frame_rate, (w, h))

# 画像処理
while cap.isOpened():
  ret, img = cap.read()
  if not ret:
    break
  applied_img = apply_color_tone(img)
  out.write(apply_blur(applied_img))

# ファイルの解放
cap.release()
out.release()

play_video("movie_1_edited.mp4", 240)

In [ ]:
#@title 第12回予定 (8) 本 p.175 (5-5-2) と同様に動画の色変換とエッジ強調 + 自動再生

import cv2
import numpy as np

def apply_color_tone(img):
  filter = np.array([
    [5, 0, 0],
    [0, 5, 0],
    [0, 0, 5]
  ])
  applied_img = cv2.transform(img, filter)

  # 値を0〜255の範囲に変更
  return np.clip(applied_img, 0, 255).astype(np.uint8)

def apply_edges(img):
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  edges = cv2.Canny(gray, 100, 200)

  # エッジを太くする
  kernel = np.ones((3, 3), np.uint8)
  edges_dilated = cv2.dilate(edges, kernel, iterations=1)

  # エッジを黒色で描画
  img[edges_dilated == 255] = (0, 0, 0)

  return img

# 動画ファイルの読み込み
cap = cv2.VideoCapture("movie.mp4")

# フレームレートの取得
frame_rate = cap.get(cv2.CAP_PROP_FPS)

# 動画の幅と高さを取得
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 動画保存条件
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("movie_2_edited.mp4", fourcc, frame_rate, (w, h))

# 画像処理
while cap.isOpened():
  ret, img = cap.read()
  if not ret:
    break

  applied_image = apply_color_tone(img)
  applied_image = apply_edges(applied_image)
  out.write(applied_image)

# ファイルの解放
cap.release()
out.release()

play_video("movie_2_edited.mp4", 240)

In [ ]:
#@title 第12回予定 (9) Chapter 5 最後：動画撮影〜色変換とエッジ強調〜自動再生をまとめて

from IPython.display import HTML
from base64 import b64encode
import cv2
import numpy as np

def apply_color_tone(img):
  filter = np.array([
    [5, 0, 0],
    [0, 5, 0],
    [0, 0, 5]
  ])
  applied_img = cv2.transform(img, filter)

  # 値を0〜255の範囲に変更
  return np.clip(applied_img, 0, 255).astype(np.uint8)

def apply_edges(img):
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  edges = cv2.Canny(gray, 100, 200)

  # エッジを太くする
  kernel = np.ones((3, 3), np.uint8)
  edges_dilated = cv2.dilate(edges, kernel, iterations=1)

  # エッジを黒色で描画
  img[edges_dilated == 255] = (0, 0, 0)

  return img

# PCカメラから動画ファイルを保存し読み込み
frame_rate = 30
duration = 10
VideoWriter("movie_3.mp4", duration, frame_rate, (640, 480))
cap = cv2.VideoCapture("movie_3.mp4")

# フレームレートの取得
frame_rate = cap.get(cv2.CAP_PROP_FPS)
# 動画の幅と高さを取得
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 編集した動画の保存条件
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("movie_3_edited.mp4", fourcc, frame_rate, (w, h))

# 動画編集
while cap.isOpened():
  ret, img = cap.read()
  if not ret:
    break

  # 画像処理を実行
  applied_image = apply_color_tone(img)
  applied_image = apply_edges(applied_image)
  out.write(applied_image)

# ファイルの解放
cap.release()
out.release()

play_video("movie_3_edited.mp4", 240)
